# **Forgery detection**

In [ ]:
!pip install kagglehub tqdm

import kagglehub
import os

# newest dataset
path = kagglehub.dataset_download("divg07/casia-20-image-tampering-detection-dataset")
path_comofod = kagglehub.dataset_download("tusharchauhan1898/comofod")

# path to CASIA2 directory, needed for casia_dataset()
DATA_PATH = os.path.join(path, "CASIA2")

# path to COMOFOD directory
DATA_PATH_COMOFOD = os.path.join(path_comofod, "CoMoFoD_small_v2")

print("Path to dataset CASIA2:", DATA_PATH)
print("Path to dataset COMOFOD:", DATA_PATH_COMOFOD)

**Needed libraries**

In [ ]:
import os
import io
# images
import cv2 # pip install opencv-python
import PIL
# stats
import numpy as np
import pandas as pd
from scipy.stats import skew
# plots
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns # pip install seaborn
# LBP & Max Pooling
from skimage.feature import local_binary_pattern
from skimage.measure import block_reduce
# image processing
from sklearn.preprocessing import StandardScaler # pip install scikit-learn
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split # dataset split learning/testing
# classifiers
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
# raports
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
# loading bar
from tqdm.notebook import tqdm
# SHapley Additive exPlanations
import shap

**Constants declaration**

*   IMAGE_SIZE - resolution for all images,
*   FEATURE_NAMES:
    *  ELA:
        * mean x3 colors,
        * standard deviation x3 colors.
    *  mean for base image x3 colors,
    *  standard deviation for base image x3 colors,
    *  skewness x3 colors,
    *  Local Binary Patterns for 10 bins.



In [ ]:
# base resolution
IMAGE_SIZE = (250, 250)
# all 25 features sorted
FEATURE_NAMES = ['ELA_R_mean', 'ELA_G_mean', 'ELA_B_mean', 'ELA_R_std', 'ELA_G_std',
                 'ELA_B_std','Mean_B', 'Mean_G', 'Mean_R', 'Std_B', 'Std_G', 'Std_R',
                 'Skew_B', 'Skew_G', 'Skew_R'] + [f'LBP_bin_{i}' for i in range(10)]

**Error Level Analysis**

It saves the image at a specified JPEG quality (default 90%) and then calculates the difference between the original and the compressed version to detect areas with different compression levels. If a fragment of the image was pasted from another file, it will have a different noise structure than the rest of the image. Max Pooling is used to summarize these differences and derive statistics: mean and standard deviation.

In [ ]:
def ELA(img_path, quality=90):
    try:
        # loading the original photo and processing it using a buffer (no temporary file)
        original = PIL.Image.open(img_path).convert('RGB')
        # rescale to IMAGE_SIZE
        original = original.resize(IMAGE_SIZE)

        buffer = io.BytesIO()
        original.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        compressed = PIL.Image.open(buffer)

        # ELA calculation
        ela_image = PIL.ImageChops.difference(original, compressed)
        ela_array = np.array(ela_image)

        # Max Pooling 5x5 block size
        ela_pooled = block_reduce(ela_array, block_size=(5, 5, 1), func=np.max)

        # mean and std after Pooling, for every RGB color
        ela_mean = np.mean(ela_pooled, axis=(0, 1))
        ela_std = np.std(ela_pooled, axis=(0, 1))

        return np.concatenate([ela_mean, ela_std])
    except Exception as e:
        return None

A modified version of the previous ELA function to show where there are noticeable differences in the noise structure of the image.

In [ ]:
def ELA_example(img_path, quality=90, scale=15):
    # loading the original photo and processing it using a buffer (no temporary file)
    original = PIL.Image.open(img_path).convert('RGB')
    # rescale to IMAGE_SIZE
    original = original.resize(IMAGE_SIZE)

    buffer = io.BytesIO()
    original.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    compressed = PIL.Image.open(buffer)

    # ELA calculation
    ela_image = PIL.ImageChops.difference(original, compressed)

    # minimum and maximum brightness search for each RGB color
    extrema = ela_image.getextrema()

    # for each point, the first index (extremum) is selected in each of the three RGB colors
    max_diff = max([ex[1] for ex in extrema])

    # solving the problem of division by 0 when there are two identical images
    if max_diff == 0:
        max_diff = 1

    # multiplying each point by the scale to show differences in the resulting image
    ela_image = ela_image.point(lambda p: p * scale)

    plt.figure()
    plt.imshow(ela_image)
    plt.title('Error Level Analysis')
    plt.axis('off')
    plt.show()

**Image Features Function**

Combines the results of several methods:

* retrieves data from the ELA function,

* calculates color statistics (mean, standard deviation, skewness) in RGB space,

* uses Local Binary Patterns to analyze image texture.


In [ ]:
def image_features(img_path):
    features = []

    # ELA 
    ela_features = ELA(img_path)
    if ela_features is None: return None # error type if sth
    features.extend(ela_features)

    # image load
    img = cv2.imread(img_path)
    if img is None: return None # error type if sth

    # rescale to IMAGE_SIZE
    img = cv2.resize(img, IMAGE_SIZE)

    # mean and std
    mean_c, std_c = cv2.meanStdDev(img)
    # skewness
    skewness = skew(img.reshape(-1, 3), axis=0)
    # add them to the list
    features.extend(mean_c.flatten())
    features.extend(std_c.flatten())
    features.extend(skewness)

    # LBP, 8 neighbours, R=1
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    # 10 bins stats
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), density=True)
    features.extend(lbp_hist)

    return np.array(features)

**Dataset CASIA2**

source: https://www.kaggle.com/datasets/divg07/casia-20-image-tampering-detection-dataset?select=CASIA2

In [ ]:
def casia_dataset(base_path, max_images=None):
    # X - input/features, y - authentic/tampered
    X, y = [], []
    classes = {'Au': 0, 'Tp': 1}

    for class_name, label in classes.items():
        folder_path = os.path.join(base_path, class_name)
        if not os.path.exists(folder_path):
            print(f"Missing directory: {folder_path}")
            continue

        print(f"Computing directory: {class_name}...")
        filenames = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp'))]

        if max_images:
            filenames = filenames[:max_images]

        for filename in tqdm(filenames):
            img_path = os.path.join(folder_path, filename)
            features = image_features(img_path)

            if features is not None:
                X.append(features)
                y.append(label)

    return np.array(X), np.array(y)

**Visualization function for a modified image with a mask applied**

Displays the image side-by-side, a black-and-white mask indicating the manipulation, and applies the mask to the image (as a red area).

In [ ]:
def visualize_tampering(image_path, mask_path):
    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        print("Files not found.")
        return

    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    tp_ex = {'image': img, 'mask': mask, 'image_path': image_path}

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(tp_ex['image'])
    axes[0].set_title('Tampered image', fontsize=14)
    axes[0].axis('off')

    axes[1].imshow(tp_ex['mask'], cmap='gray')
    axes[1].set_title('Mask', fontsize=14)
    axes[1].axis('off')

    plt.suptitle(os.path.basename(tp_ex['image_path']), fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

    # mask to image
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    overlay = tp_ex['image'].copy()
    binary = tp_ex['mask'] > 127

    # red mask
    overlay[binary] = [255, 0, 0]
    blended = (0.55 * tp_ex['image'] + 0.45 * overlay).astype(np.uint8)

    ax.imshow(blended)
    red_patch = patches.Patch(color='red', label='tampered area')
    ax.legend(handles=[red_patch], loc='lower right')
    ax.set_title('Mask used on tampered image', fontsize=14)
    ax.axis('off')
    plt.show()

**Confusion Table Function**

Represents the performance of the classification algorithm (classifier). Each row of the table represents the possible actual labels of the units being tested, and each column represents the labels predicted by the algorithm.

In [ ]:
def confusion_matrices(X_test, y_test, classifiers):
    # number of classifiers
    n_models = len(classifiers)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))

    for ax, (name, clf) in zip(axes, classifiers.items()):
        # prediction
        y_pred = clf.predict(X_test)
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

        # heatmap
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False, xticklabels=['pozytywna (0)', 'negatywna (1)'], yticklabels=['pozytywna (0)', 'negatywna (1)'])

        ax.set_title(f'Klasyfikator: {name}', fontsize=14)
        ax.set_xlabel('predykcja')
        ax.set_ylabel('rzeczywistość')

    plt.tight_layout()
    plt.show()

**Image Resolution Histogram Function**

The image collection is analyzed for image height and width.

In [ ]:
def image_resolutions(base_path, label_name):
    data = []
    for filename in os.listdir(base_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.tif', '.png', '.bmp')):
            img_path = os.path.join(base_path, filename)

            # only height and width
            img = cv2.imread(img_path)
            if img is not None:
                h, w, _ = img.shape
                data.append({
                    'width': w,
                    'height': h,
                    'class': label_name
                })
    return data

**Tampered image example**

Image *Tp_D_CNN_M_N_ani00023_ani00024_10205.tif* and its mask *Tp_D_CNN_M_N_ani00023_ani00024_10205_gt.png*

In [ ]:
file_name = "Tp_D_CNN_M_N_ani00023_ani00024_10205"

# path building
img_path = os.path.join(DATA_PATH, "Tp", f"{file_name}.tif")
mask_path = os.path.join(DATA_PATH, "CASIA 2 Groundtruth", f"{file_name}_gt.png")

visualize_tampering(img_path, mask_path)

**Error Level Analysis example after image rescale**

In [ ]:
ELA_example(img_path)

**Image Resolution Histograms in CASIA2**

The histograms feature bars grouping similar values, and a kernel density estimator (KDE), a type of nonparametric estimator designed to determine the density distribution of a random variable based on the obtained sample. It determines the values ​​of the variable under study during previous measurements.

The conclusion from this study is that the images have different resolutions, and the distribution is divergent between authentic and faked images, which requires rescaling the entire set.

In [ ]:
# datasets paths
res_au = image_resolutions(os.path.join(DATA_PATH, "Au"), 'Authentic')
res_tp = image_resolutions(os.path.join(DATA_PATH, "Tp"), 'Tampered')

# height and width in one dataframe
df_res = pd.DataFrame(res_au + res_tp)

sns.set_theme(style="whitegrid")

# plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# width
sns.histplot(data=df_res, x='width', hue='class', kde=True, element="step", palette='viridis', ax=axes[0], alpha=0.5)
axes[0].set_title('Image width histogram', fontsize=14, fontweight='bold')
axes[0].set_xlabel('width (px)')
axes[0].set_ylabel('images count')

# height
sns.histplot(data=df_res, x='height', hue='class', kde=True, element="step", palette='magma', ax=axes[1], alpha=0.5)
axes[1].set_title('Image height histogram', fontsize=14, fontweight='bold')
axes[1].set_xlabel('height (px)')
axes[1].set_ylabel('images count')

plt.tight_layout()
plt.show()

**Main for CASIA2**

At the beginning, the RANDOM_NUMBER variable determines the randomness of the selected images, split into training and test sets. MAX_IMAGES was used only during program development to avoid working with the entire set every time. Removing it will run all images.

Exploratory Data Analysis is used to evaluate which set the model will work on. An 80/20 split (training/testing) was applied.

Next, for `PCA_USED = True`, principal component analysis (PCA) was used to reduce features that did not meet the 95% most important criteria.

Three different classifiers are trained:

* Random Forest, for 100 decision trees, depending on the random number declared at the beginning.

It bases its decisions on the collective wisdom of many individual decision trees. During the training process, the algorithm creates a large number of trees from random subsets of data and features, so each of them views the problem slightly differently. The final classification is the result of a democratic vote, as the system selects the class indicated by the majority of the trees.

* Support Vector Machine (SVM)

Strives to find the optimal boundary (hyperplane) separating different data groups. It searches for a dividing line that provides the maximum margin between classes. The closest points to these classes are called support vectors. It can project data to higher dimensions, allowing for the efficient separation of very complex and nonlinear sets of information.

* k-Nearest Neighbors, for 5 neighbors.

Based on the assumption that similar objects are typically located close to each other in feature space, it stores all input data. When a new point appears, k-NN calculates its distance from all known examples and assigns it the class that dominates among its k nearest neighbors.

---

**Finally, we receive a report on the model's performance on the test data:**

* Accuracy - % of correct decisions, compared in a column chart depending on the classifier used,
* Recall - sensitivity to modified photos,
* Precision - how many suspected fake photos were correctly detected,
* F1-Score - harmonic mean between Precision and Recall sensitivity,
* Support - number of occurrences.

---

**Pipeline**

1. Before the image is analyzed, it is loaded and transformed to a fixed resolution of `IMAGE_SIZE = (250, 250)`. This prevents the model from learning noise resulting from variable image dimensions.

2. Feature extraction - 25 per image:

    * Compression artifacts (ELA - 6 features): The image is compressed to JPEG format with 90% quality. The pixel-level difference between the original and the compressed image is calculated. The difference image is passed through Max Pooling with a 5x5 window. From this, the mean and standard deviation are calculated for each of the 3 RGB channels.

    * Base image statistics (9 features): The mean brightness, standard deviation, and skewness of the color distribution are calculated separately for each channel.

    * Local Binary Patterns texture (10 features): The image is converted to grayscale. LBP is run for a radius of 1 and 8 neighbors. The result is compressed into a 10-bin histogram illustrating the frequency of edges, corners, and flat areas.

3. The dataset is split into a training and test set 80/20. In the test phase, the algorithm uses fake and real images in a 1:1 ratio, which protects the Accuracy metric from being biased by the dominance of one class.

4. The extracted features have drastically different ranges (e.g., skewness can have values ​​around 1-2, while the sum of the LBP bins can have completely different values). They are rescaled around zero with a standard deviation of 1, which is required, especially for the correct performance of k-NN and SVM classifiers.

5. **(two versions: with and without this point)** PCA is performed on 25 features to reduce those that fall outside the 95% variance.

6. Three classifiers: Random Forest, SVM, and k-NN learn the relationships between features and the Au/Tp label. They are then tested on an independent, balanced test set.

7. Finally, the algorithm evaluates the Random Forest based on why it made a given decision. SHAP displays all features from most important to least important. For example, a dot shifted to the right for a given feature indicates that its high value strongly biased the result toward classifying the image as fake. Due to its computational complexity, this classifier was chosen over SHAP.



---

**Main with PCA**

---



In [ ]:
RANDOM_NUMBER = 42
MAX_IMAGES = 50

print("Data loading...")
X, y = casia_dataset(DATA_PATH)

if len(X) == 0:
    print("No data loaded.")
else:
    # table with features matrix and labels
    df = pd.DataFrame(X, columns=FEATURE_NAMES)
    df['label'] = y
    class_count = df['label'].value_counts().values

    # EDA
    print("\nExploratory Data Analysis (EDA)")
    plt.figure()
    plt.bar(['Authentic (0)', 'Tampered (1)'], class_count)
    plt.title('Class count in dataset')
    plt.show()

    # test set balancing
    au_idx = np.where(y == 0)[0]
    tp_idx = np.where(y == 1)[0]

    # how many images for test, 20% of smaller class
    test_size_per_class = int(min(len(au_idx), len(tp_idx)) * 0.2)

    # picking images from each class
    np.random.default_rng(RANDOM_NUMBER)
    test_au = np.random.choice(au_idx, test_size_per_class, replace=False)
    test_tp = np.random.choice(tp_idx, test_size_per_class, replace=False)

    test_indices = np.concatenate([test_au, test_tp])
    train_indices = np.setdiff1d(np.arange(len(y)), test_indices)

    X_train, X_test = df.iloc[train_indices].drop('label', axis=1), df.iloc[test_indices].drop('label', axis=1)
    y_train, y_test = y[train_indices], y[test_indices]

    print(f"\nBalanced test dataset: {len(test_au)} Au (0), {len(test_tp)} Tp (1)")

    # rescale of features for classifiers
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # conversion to dataframe for SHAP
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURE_NAMES)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES)

    pca = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    print(f"\nPCA: Reduced from {X_train_scaled.shape[1]} to {X_train_pca.shape[1]} features.")

    classifiers = {
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_NUMBER),
        "SVM": SVC(kernel='rbf', random_state=RANDOM_NUMBER),
        "k-NN": KNeighborsClassifier(n_neighbors=5)
    }

    print("\nClassifiers results")
    clf_names = []
    clf_accuracies = []

    for name, clf in classifiers.items():
        # PCA
        clf.fit(X_train_pca, y_train)
        y_pred = clf.predict(X_test_pca)

        acc = accuracy_score(y_test, y_pred)
        clf_names.append(name)
        clf_accuracies.append(acc)

        print(f"\nModel: {name}")
        print(f"Accuracy: {acc:.4f}")
        print(classification_report(y_test, y_pred, target_names=['Authentic', 'Tampered']))

    plt.figure(figsize=(8, 6))
    sns.barplot(x=clf_names, y=clf_accuracies, palette="viridis", hue=clf_names, legend=False)
    plt.title('Accuracy comparison for classifiers')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)

    for index, value in enumerate(clf_accuracies):
        plt.text(index, value + 0.02, f'{value:.2f}', ha='center')
    plt.show()

**Confusion matrices**

In [ ]:
confusion_matrices(X_test_pca, y_test, classifiers)

**SHAP**

Shapley Additive exPlanations is a method for explaining model results that allows us to visualize which features play a key role in classification. It is based on game theory (Shapley values), treating each image feature as a player in a team working towards a final result.

In [ ]:
# SHAP for RF
rf_model = classifiers["Random Forest"]
explainer = shap.TreeExplainer(rf_model)

shap_values = explainer.shap_values(X_test_pca)
shap_vals_to_plot = shap_values[..., 1] if len(shap_values.shape) > 2 else shap_values
shap.summary_plot(shap_vals_to_plot, X_test_pca, feature_names=FEATURE_NAMES)



---

**Main without PCA**

---



In [ ]:
RANDOM_NUMBER = 42
MAX_IMAGES = 50

print("Data loading...")
X, y = casia_dataset(DATA_PATH)

if len(X) == 0:
    print("No data loaded.")
else:
    # table with features matrix and labels
    df = pd.DataFrame(X, columns=FEATURE_NAMES)
    df['label'] = y
    class_count = df['label'].value_counts().values

    # EDA
    print("\nExploratory Data Analysis (EDA)")
    plt.figure()
    plt.bar(['Authentic (0)', 'Tampered (1)'], class_count)
    plt.title('Class count in dataset')
    plt.show()

    # test set balancing
    au_idx = np.where(y == 0)[0]
    tp_idx = np.where(y == 1)[0]

    # how many images for test, 20% of smaller class
    test_size_per_class = int(min(len(au_idx), len(tp_idx)) * 0.2)

    # picking images from each class
    np.random.default_rng(RANDOM_NUMBER)
    test_au = np.random.choice(au_idx, test_size_per_class, replace=False)
    test_tp = np.random.choice(tp_idx, test_size_per_class, replace=False)

    test_indices = np.concatenate([test_au, test_tp])
    train_indices = np.setdiff1d(np.arange(len(y)), test_indices)

    X_train, X_test = df.iloc[train_indices].drop('label', axis=1), df.iloc[test_indices].drop('label', axis=1)
    y_train, y_test = y[train_indices], y[test_indices]

    print(f"\nBalanced test dataset: {len(test_au)} Au (0), {len(test_tp)} Tp (1)")

    # rescale of features for classifiers
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # conversion to dataframe for SHAP
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURE_NAMES)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES)

    classifiers = {
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_NUMBER),
        "SVM": SVC(kernel='rbf', random_state=RANDOM_NUMBER),
        "k-NN": KNeighborsClassifier(n_neighbors=5)
    }

    print("\nClassifiers results")
    clf_names = []
    clf_accuracies = []

    for name, clf in classifiers.items():
        # without PCA
        clf.fit(X_train_scaled, y_train)
        y_pred = clf.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)
        clf_names.append(name)
        clf_accuracies.append(acc)

        print(f"\nModel: {name}")
        print(f"Accuracy: {acc:.4f}")
        print(classification_report(y_test, y_pred, target_names=['Authentic', 'Tampered']))

    plt.figure(figsize=(8, 6))
    sns.barplot(x=clf_names, y=clf_accuracies, palette="viridis", hue=clf_names, legend=False)
    plt.title('Accuracy comparison for classifiers')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)

    for index, value in enumerate(clf_accuracies):
        plt.text(index, value + 0.02, f'{value:.2f}', ha='center')
    plt.show()

**Confusion matrices**

In [ ]:
confusion_matrices(X_test_scaled, y_test, classifiers)

**SHAP**

In [ ]:
# SHAP for RF
rf_model = classifiers["Random Forest"]
explainer = shap.TreeExplainer(rf_model)

shap_values = explainer.shap_values(X_test_scaled)
shap_vals_to_plot = shap_values[..., 1] if len(shap_values.shape) > 2 else shap_values
shap.summary_plot(shap_vals_to_plot, X_test_scaled, feature_names=FEATURE_NAMES)



---


**CoMoFod dataset research**

[CoMoFod](https://www.kaggle.com/datasets/tusharchauhan1898/comofod/data)


---



**Function for loading the CoMoFod dataset for research**

In [ ]:
def comofod_dataset(base_path, max_images=None):
    # X - input/features, y - labels authentic/tampered
    X, y = [], []
    au_paths = []
    tp_paths = []
    mask_paths = []

    if not os.path.exists(base_path):
        print(f"Directory {base_path} does not exist!")

    for filename in os.listdir(base_path):
        if not filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            continue

        name, _ = os.path.splitext(filename)

        # O - original
        # B - mask
        # F - tampered image
        if name.endswith('_O'):
            au_paths.append(os.path.join(base_path, filename))
        elif name.endswith('_B') or 'mask' in name.lower():
            mask_paths.append(os.path.join(base_path, filename))
        elif '_F' in name:
            tp_paths.append(os.path.join(base_path, filename))

    # limits
    if max_images:
        au_paths = au_paths[:max_images]
        tp_paths = tp_paths[:max_images]

    print("\nclass processing: Authentic")
    for img_path in tqdm(au_paths, desc="klasa Au"):
        features = image_features(img_path)
        if features is not None:
            X.append(features)
            y.append(0)

    print("\nclass processing: Tampered")
    for img_path in tqdm(tp_paths, desc="klasa Tp"):
        features = image_features(img_path)
        if features is not None:
            X.append(features)
            y.append(1)

    return np.array(X), np.array(y)

**Example image visualisation from CoMoFod dataset**

In [ ]:
file_name = "002"

# path building
img_path = os.path.join(DATA_PATH_COMOFOD, f"{file_name}_F.png")
mask_path = os.path.join(DATA_PATH_COMOFOD, f"{file_name}_B.png")

visualize_tampering(img_path, mask_path)

In [ ]:
file_name = "002"

original = cv2.cvtColor(cv2.imread(os.path.join(DATA_PATH_COMOFOD, f"{file_name}_O.png")), cv2.COLOR_BGR2RGB)
modified = cv2.imread(os.path.join(DATA_PATH_COMOFOD, f"{file_name}_M.png"), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original)
axes[0].set_title('Original', fontsize=14)
axes[0].axis('off')

axes[1].imshow(modified)
axes[1].set_title('Modified area', fontsize=14)
axes[1].axis('off')

plt.suptitle(file_name, fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**Error Level Analysis for CoMoFod**

In [ ]:
ELA_example(img_path)

**Main**

In [ ]:
RANDOM_NUMBER = 42
MAX_IMAGES = 50

print("Data loading...")
X, y = comofod_dataset(DATA_PATH_COMOFOD, 200)

if len(X) == 0:
    print("No data loaded.")
else:
    # table with features matrix and labels
    df = pd.DataFrame(X, columns=FEATURE_NAMES)
    df['label'] = y
    class_count = df['label'].value_counts().values

    # EDA
    print("\nExploratory Data Analysis (EDA)")
    plt.figure()
    plt.bar(['Authentic (0)', 'Tampered (1)'], class_count)
    plt.title('Class count in dataset')
    plt.show()

    # test set balancing
    au_idx = np.where(y == 0)[0]
    tp_idx = np.where(y == 1)[0]

    # how many images for test, 20% of smaller class
    test_size_per_class = int(min(len(au_idx), len(tp_idx)) * 0.2)

    # picking images from each class
    np.random.default_rng(RANDOM_NUMBER)
    test_au = np.random.choice(au_idx, test_size_per_class, replace=False)
    test_tp = np.random.choice(tp_idx, test_size_per_class, replace=False)

    test_indices = np.concatenate([test_au, test_tp])
    train_indices = np.setdiff1d(np.arange(len(y)), test_indices)

    X_train, X_test = df.iloc[train_indices].drop('label', axis=1), df.iloc[test_indices].drop('label', axis=1)
    y_train, y_test = y[train_indices], y[test_indices]

    print(f"\nBalanced test dataset: {len(test_au)} Au (0), {len(test_tp)} Tp (1)")

    # rescale of features for classifiers
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # conversion to dataframe for SHAP
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURE_NAMES)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES)

    classifiers = {
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_NUMBER),
        "SVM": SVC(kernel='rbf', random_state=RANDOM_NUMBER),
        "k-NN": KNeighborsClassifier(n_neighbors=5)
    }

    print("\nClassifiers results")
    clf_names = []
    clf_accuracies = []

    for name, clf in classifiers.items():
        # without PCA
        clf.fit(X_train_scaled, y_train)
        y_pred = clf.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)
        clf_names.append(name)
        clf_accuracies.append(acc)

        print(f"\nModel: {name}")
        print(f"Accuracy: {acc:.4f}")
        print(classification_report(y_test, y_pred, target_names=['Authentic', 'Tampered']))

    # wykres kolumnowy porównawczy
    plt.figure(figsize=(8, 6))
    sns.barplot(x=clf_names, y=clf_accuracies, palette="viridis", hue=clf_names, legend=False)
    plt.title('Accuracy comparison for classifiers')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)

    for index, value in enumerate(clf_accuracies):
        plt.text(index, value + 0.02, f'{value:.2f}', ha='center')
    plt.show()

**Confusion matrices**

In [ ]:
confusion_matrices(X_test_scaled, y_test, classifiers)

**SHAP**

In [ ]:
# SHAP for RF
rf_model = classifiers["Random Forest"]
explainer = shap.TreeExplainer(rf_model)

shap_values = explainer.shap_values(X_test_scaled)
shap_vals_to_plot = shap_values[..., 1] if len(shap_values.shape) > 2 else shap_values
shap.summary_plot(shap_vals_to_plot, X_test_scaled, feature_names=FEATURE_NAMES)